

# Notebook 



-----------

Before running this notebook, take a look at the _settings.py file. This python script defines paths to data files and directories used in this notebook.
It also defines some configurations, resolutions and options to use when running the script.
Adjust the paths as needed before proceeding with the rest of the notebook.

In [1]:
# Let's start by importing all necessary packages and functions defined in this folder.

import numpy as np
import xarray as xr
import pandas as pd
import geopandas as gpd
import pickle as pk
from scipy import interpolate
import regionmask
import glob, os, re, sys
import openpyxl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import warnings
from math import ceil 
warnings.filterwarnings('ignore')

pd.set_option('display.max_rows', 20)
%matplotlib inline 

import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature

sys.path.append('../..') # location of dem4cli package
#import demographics4climate as dem4cli # to call them as dem4cli.function 
from dem4cli import * # to call fxns directly


ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/scratch/brussel/vo/000/bvo00012/vsc11359/dem4cli/dem4cli/lib/python3.11/site-packages/traitlets/traitlets.py", line 632, in get
    value = obj._trait_values[self.name]
            ~~~~~~~~~~~~~~~~~^^^^^^^^^^^
KeyError: '_control_lock'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/scratch/brussel/vo/000/bvo00012/vsc11359/dem4cli/dem4cli/lib/python3.11/site-packages/zmq/eventloop/zmqstream.py", line 565, in _log_error
    f.result()
  File "/scratch/brussel/vo/000/bvo00012/vsc11359/dem4cli/dem4cli/lib/python3.11/site-packages/ipykernel/kernelbase.py", line 340, in dispatch_control
    async with self._control_lock:
               ^^^^^^^^^^^^^^^^^^
  File "/scratch/brussel/vo/000/bvo00012/vsc11359/dem4cli/dem4cli/lib/python3.11/site-packages/traitlets/traitlets.py", line 687, in __get__
    return t.cast(G, self.ge

In [2]:
# This code can be run using different configurations, different resolutions and different options. Which ones are we using?

flags

{'version': 2,
 'pop_resolution': 0.1,
 'GMT_mapping': 'year_to_year',
 'cohort_sizes_source': 'UNWPP2024',
 'countrymask': 'shapefile'}

In [3]:
# We'll run the analysis on a domain that can be specified here: Lat S, Lat N, Lon W, Lon E.

bbox_indiaws = [ 2.00, 40.00, 66.00, 100.00 ]
bbox_europe = [ 31.99,  71.09, -14.96,  34.94]

## Population preprocessing

This section take care of loading and preprocessing all demographic data required for the analysis. 
It uses a wrapper function called preprocess_all_country_data and defined in population_demographics.py.

You can have a look at how preprocess_all_country_data is defined to understand which functions it calls and what it does. 
You can also read the Methods part of Grant et al. (2025) or the Supplementary Material of Thiery et al. (2021) to understand what demographic data are used in this analysis and how they are processed for this purpose.

In [4]:
# Make sure that the function below call the shapefile you have just created.
# Make sure that the path to data directories are correctly defined to be able to load the required data.

d_countries = preprocess_all_country_data(

    filepath_lifeexpectancy = filepath_lifeexpectancy, # life expectancy data
    start_birthyear=2020,
    end_birthyear=2025,                 # endyear is taken from end_birthyear + max life expectancy

    dir_cohortsizes = dir_cohortsizes,  # cohort size data
    data_source_cohorts='UNWPP2024',
    extend_method='linear',             # note, 'slinear' not implemented for UNWPP2024
    by_sex=False,                       # NOTE by_sex not implemented
                                            
    dir_population= dir_population,     # gridded pop data 
    ssp=2,
    urbanrural=False,                   # NOTE urbanrural not implemented for v2
    bbox = bbox_europe,

    filepath_countrymask = filepath_countrymask,
    data_source_countrymask = 'shapefile',
    fillcoast=False, 
    fix_smallislands=False,
    
    filepath_world_bank = filepath_world_bank_meta, # metadata 
    filepath_lookuptable = filepath_lookuptable,    # country filtering
    filter_countries=True,
    worldbank_filter=True, 
    )

df_countries = d_countries['info_pop']
gdf_country_borders = d_countries['borders'] 
da_regions = df_countries['region'].unique()
da_population = d_countries['population_map']
df_birthyears = d_countries['birth_years'] # NA
df_life_expectancy_5 = d_countries['life_expectancy_5']
da_cohort_size = d_countries['cohort_size']
countries_regions, countries_mask = d_countries['mask']

load_country_metadata took 0.06 s
load_unwpp_lifeexpectancy took 22.90 s
get_life_expectancies took 0.00 s
loading cohort sizes from UNWPP2024
load_cohort_sizes took 23.08 s
interpolate_cohortsize_countries took 0.01 s
opening compass - historical
opening compass - ssp2
load_population took 1.86 s
load_countrymask took 2.30 s
preprocess_all_country_data took 50.25 s


In [12]:
df_life_expectancy_5

Country,Albania,Algeria,Andorra,Austria,Belarus,Belgium,Bosnia and Herzegovina,Bulgaria,Croatia,Cyprus,...,Serbia,Slovakia,Slovenia,Spain,Sweden,Switzerland,Tunisia,Türkiye,Ukraine,United Kingdom
Year,,,,,,,,,,,,,,,,,,,,,
2020,86.6001,84.2175,90.8199,88.5604,81.0125,88.7051,84.6465,82.395,85.2238,88.2251,...,83.49,85.0173,88.0752,90.19,89.7724,90.5269,83.7551,84.6375,81.2976,87.935
2021,86.7605,84.3678,90.9386,88.7147,81.1799,88.8552,84.8188,82.5465,85.3918,88.3808,...,83.6672,85.1695,88.2295,90.3076,89.9191,90.6576,83.9189,85.1218,81.44,88.0767
2022,86.923,84.5161,91.0655,88.8823,81.3517,88.9993,84.9906,82.6858,85.5442,88.5442,...,83.8383,85.3294,88.3943,90.4504,90.0557,90.7919,84.0811,85.2946,81.5875,88.2177
2023,87.0848,84.6631,91.1889,89.044,81.5236,89.1554,85.1543,82.8324,85.7104,88.712,...,84.0101,85.4824,88.5423,90.5779,90.1884,90.9169,84.2544,85.4623,81.7357,88.3511
2024,87.2462,84.8186,91.3063,89.2032,81.6853,89.2992,85.3219,82.9786,85.8653,88.8691,...,84.1825,85.6319,88.6981,90.7083,90.3242,91.0416,84.4182,85.6319,81.887,88.4992
2025,87.3978,84.9681,91.4303,89.3621,81.8532,89.4491,85.4978,83.1273,86.0255,89.0246,...,84.353,85.7974,88.8321,90.8387,90.4485,91.175,84.5982,85.8005,82.0306,88.6532


## Land fraction exposed



In [5]:
# Load results from RIME-X

In [6]:
# PLOT results from RIME-X


## Lifetime exposure 




In [7]:
# Call function that processes results from RIME-X

In [8]:
# Plot results for lifetime exposure using RIME-X results



## Compute multi-model mean and model spread


## Plot results for the multi-model mean and the model spread